# Review: `romanisim_l3_pipeline.py`

Inspect the execution of the L3 (co-added mosaic) cutout pipeline and visualize its results.

The L3 pipeline writes to `<data_dir>/<pipeline_label>/05_romanisim/sca##/`:

- `Exposure_<label>_<uid>_<band>.npy` — 73x73 cutouts in mosaic-native **MJy/sr** surface brightness (NaN-padded where the cutout runs off the mosaic edge).
- `sca##_<band>_batch#_mosaic.png` — a diagnostic image of each co-added L3 mosaic.
- `batch_complete_sca##_<band>_batch#.txt` — a per-batch sentinel whose contents is the number of systems in that batch.
- `_work_sca##_<band>_batch#/` — a temporary L2/asn/coadd working dir that the pipeline deletes on success. **A leftover `_work_` dir means that batch crashed or was interrupted.**


In [ ]:
import os
from glob import glob
from collections import defaultdict

import yaml
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import mejiro

# ---- choose the config the pipeline was run with --------------------------
config_name = 'roman_data_challenge_rung_1.yaml'
config_file = os.path.join(os.path.dirname(os.path.dirname(mejiro.__file__)), 'projects', 'roman_data_challenge', config_name)
with open(config_file) as f:
    config = yaml.load(f, Loader=yaml.SafeLoader)
if config['dev']:
    config['pipeline_label'] += '_dev'

label = config['pipeline_label']
l3_dir = os.path.join(config['data_dir'], label, '05_romanisim')
synth_step = '04_jax' if config['jaxtronomy']['use_jax'] else '04'
input_dir = os.path.join(config['data_dir'], label, synth_step)
bands = config['synthetic_image']['bands']

# ---- where figures from this notebook get written -------------------------
# anchored to the repo (not the kernel's cwd, which varies by how it was launched)
repo_root = os.path.dirname(os.path.dirname(mejiro.__file__))
fig_dir = os.path.join(repo_root, 'notebooks', 'figures', '05_romanisim_l3')
os.makedirs(fig_dir, exist_ok=True)
FIG_DPI = 300

def save_fig(fig, name):
    path = os.path.join(fig_dir, f'{name}.png')
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight')
    print(f'saved {os.path.relpath(path, repo_root)}')

print(f'pipeline label : {label}')
print(f'L3 output dir  : {l3_dir}')
print(f'input (step {synth_step}): {input_dir}')
print(f'bands          : {bands}')
print(f'figure dir     : {fig_dir}')
assert os.path.isdir(l3_dir), f'No L3 output directory at {l3_dir}'

## 1. Execution review

What ran, what completed, and what (if anything) failed.

In [14]:
# ---- inventory every artifact the pipeline writes -------------------------
cutouts   = sorted(glob(os.path.join(l3_dir, 'sca*', 'Exposure_*.npy')))
sentinels = sorted(glob(os.path.join(l3_dir, 'sca*', 'batch_complete_*.txt')))
mosaics   = sorted(glob(os.path.join(l3_dir, 'sca*', '*_mosaic.png')))
workdirs  = sorted(glob(os.path.join(l3_dir, 'sca*', '_work_*')))

print(f'{len(cutouts):>8,} cutout (.npy) files')
print(f'{len(sentinels):>8,} completed-batch sentinels')
print(f'{len(mosaics):>8,} diagnostic mosaic PNGs')
print(f'{len(workdirs):>8,} leftover _work_ dirs  <-- nonzero => crashed/interrupted batches')

# sentinels record the per-batch system count; they should sum to (#cutouts)
sentinel_total = sum(int(open(s).read().strip() or 0) for s in sentinels)
print(f'\nsystems claimed by sentinels : {sentinel_total:,}')
print(f'cutout files on disk          : {len(cutouts):,}')
if sentinel_total != len(cutouts):
    print('  !! mismatch: completed batches do not account for every cutout on disk')

   3,900 cutout (.npy) files
      39 completed-batch sentinels
      39 diagnostic mosaic PNGs
      54 leftover _work_ dirs  <-- nonzero => crashed/interrupted batches

systems claimed by sentinels : 3,900
cutout files on disk          : 3,900


In [15]:
# ---- parse names into a (sca, band, batch) view ---------------------------
def parse_cutout(path):
    # Exposure_<label...>_<uid>_<band>.npy  -> (sca, uid, band)
    stem = os.path.basename(path)[len('Exposure_'):-len('.npy')]
    uid, band = stem.rsplit('_', 2)[-2:]
    sca = os.path.basename(os.path.dirname(path))   # 'sca01'
    return sca, uid, band

def parse_sentinel(path):
    # batch_complete_sca01_F106_batch0.txt
    name = os.path.basename(path)[len('batch_complete_'):-len('.txt')]
    sca, band, batch = name.split('_')
    return sca, band, int(batch[len('batch'):]), int(open(path).read().strip() or 0)

# completed batches table
batch_rows = sorted(parse_sentinel(s) for s in sentinels)
print(f"{'sca':<6}{'band':<7}{'batch':>6}{'systems':>10}")
print('-' * 29)
for sca, band, batch, n in batch_rows:
    print(f'{sca:<6}{band:<7}{batch:>6}{n:>10,}')

sca   band    batch   systems
-----------------------------
sca01 F106        0       100
sca01 F129        0       100
sca01 F158        0       100
sca02 F106        0       100
sca02 F129        0       100
sca02 F158        0       100
sca03 F106        0       100
sca03 F129        0       100
sca03 F158        0       100
sca04 F106        0       100
sca04 F129        0       100
sca04 F158        0       100
sca05 F106        0       100
sca05 F129        0       100
sca05 F158        0       100
sca06 F106        0       100
sca06 F129        0       100
sca06 F158        0       100
sca07 F106        0       100
sca07 F129        0       100
sca07 F158        0       100
sca09 F106        0       100
sca09 F129        0       100
sca09 F158        0       100
sca10 F106        0       100
sca10 F129        0       100
sca10 F158        0       100
sca11 F106        0       100
sca11 F129        0       100
sca11 F158        0       100
sca15 F106        0       100
sca15 F129

In [16]:
# ---- cutouts produced per SCA/band, vs. systems available in the input ----
# (reveals completion fraction: the pipeline batches systems, so a partial run
#  only covers the first capacity*n_batches systems of each SCA/band.)
produced = defaultdict(int)
for c in cutouts:
    sca, uid, band = parse_cutout(c)
    produced[(sca, band)] += 1

def input_count(sca, band):
    return len(glob(os.path.join(input_dir, sca, f'SyntheticImage_*_{band}.pkl')) +
               glob(os.path.join(input_dir, sca, f'SyntheticImage_*_{band}.npz')))

scas = sorted({k[0] for k in produced})
print(f"{'sca':<7}{'band':<7}{'produced':>10}{'available':>11}{'pct':>7}")
print('-' * 42)
for sca in scas:
    for band in bands:
        n_out = produced.get((sca, band), 0)
        if n_out == 0:
            continue
        n_in = input_count(sca, band)
        pct = 100 * n_out / n_in if n_in else float('nan')
        print(f'{sca:<7}{band:<7}{n_out:>10,}{n_in:>11,}{pct:>6.1f}%')

sca    band     produced  available    pct
------------------------------------------
sca01  F106          100      5,864   1.7%
sca01  F129          100      5,864   1.7%
sca01  F158          100      5,864   1.7%
sca02  F106          100      5,852   1.7%
sca02  F129          100      5,852   1.7%
sca02  F158          100      5,852   1.7%
sca03  F106          100      5,792   1.7%
sca03  F129          100      5,792   1.7%
sca03  F158          100      5,792   1.7%
sca04  F106          100      5,783   1.7%
sca04  F129          100      5,783   1.7%
sca04  F158          100      5,783   1.7%
sca05  F106          100      5,954   1.7%
sca05  F129          100      5,954   1.7%
sca05  F158          100      5,954   1.7%
sca06  F106          100      6,037   1.7%
sca06  F129          100      6,037   1.7%
sca06  F158          100      6,037   1.7%
sca07  F106          100      5,970   1.7%
sca07  F129          100      5,970   1.7%
sca07  F158          100      5,970   1.7%
sca09  F106

In [17]:
# ---- surface any problems -------------------------------------------------
problems = []

if workdirs:
    problems.append(f'{len(workdirs)} leftover _work_ dir(s) -> these batches did not finish cleanly:')
    for w in workdirs:
        contents = os.listdir(w)
        problems.append(f'    {os.path.relpath(w, l3_dir)}  ({len(contents)} item(s) inside)')

# SCA dirs that hold cutouts but have no sentinel at all (interrupted before completion)
sca_with_cutouts  = {parse_cutout(c)[0] for c in cutouts}
sca_with_sentinel = {parse_sentinel(s)[0] for s in sentinels}
orphan = sca_with_cutouts - sca_with_sentinel
if orphan:
    problems.append(f'SCA dir(s) with cutouts but no completion sentinel: {sorted(orphan)}')

if problems:
    print('\n'.join(problems))
else:
    print('No problems detected: every batch finished cleanly and tidied its work dir.')

54 leftover _work_ dir(s) -> these batches did not finish cleanly:
    sca01/_work_sca01_F106_batch0  (0 item(s) inside)
    sca01/_work_sca01_F129_batch0  (0 item(s) inside)
    sca01/_work_sca01_F158_batch0  (0 item(s) inside)
    sca02/_work_sca02_F106_batch0  (0 item(s) inside)
    sca02/_work_sca02_F129_batch0  (0 item(s) inside)
    sca02/_work_sca02_F158_batch0  (0 item(s) inside)
    sca03/_work_sca03_F106_batch0  (0 item(s) inside)
    sca03/_work_sca03_F129_batch0  (0 item(s) inside)
    sca03/_work_sca03_F158_batch0  (0 item(s) inside)
    sca04/_work_sca04_F106_batch0  (0 item(s) inside)
    sca04/_work_sca04_F129_batch0  (0 item(s) inside)
    sca04/_work_sca04_F158_batch0  (0 item(s) inside)
    sca05/_work_sca05_F106_batch0  (0 item(s) inside)
    sca05/_work_sca05_F129_batch0  (0 item(s) inside)
    sca05/_work_sca05_F158_batch0  (0 item(s) inside)
    sca06/_work_sca06_F106_batch0  (0 item(s) inside)
    sca06/_work_sca06_F129_batch0  (0 item(s) inside)
    sca06/_work

## 2. Cutout sanity statistics

L3 cutouts carry NaN padding at mosaic edges, so use NaN-aware stats. Flag anything empty or fully NaN.

In [18]:
# sample a manageable subset for the per-pixel statistics
rng = np.random.default_rng(0)
sample_paths = cutouts if len(cutouts) <= 2000 else \
    [cutouts[i] for i in rng.choice(len(cutouts), 2000, replace=False)]

shapes, nan_frac, peaks, totals, all_nan = [], [], [], [], 0
for p in sample_paths:
    a = np.load(p)
    shapes.append(a.shape)
    finite = np.isfinite(a)
    nan_frac.append(1.0 - finite.mean())
    if not finite.any():
        all_nan += 1
        continue
    peaks.append(np.nanmax(a))
    totals.append(np.nansum(a))

nan_frac = np.array(nan_frac); peaks = np.array(peaks); totals = np.array(totals)
print(f'inspected           : {len(sample_paths):,} cutouts (of {len(cutouts):,})')
print(f'unique shapes       : {set(shapes)}')
print(f'fully-NaN cutouts   : {all_nan}')
print(f'NaN-padded (>0 NaN) : {(nan_frac > 0).sum():,}  '
      f'(max NaN fraction in one cutout: {nan_frac.max():.1%})')
print(f'peak  MJy/sr  range : [{peaks.min():.4g}, {peaks.max():.4g}]')
print(f'total MJy/sr  range : [{totals.min():.4g}, {totals.max():.4g}]')

inspected           : 2,000 cutouts (of 3,900)
unique shapes       : {(91, 91)}
fully-NaN cutouts   : 0
NaN-padded (>0 NaN) : 0  (max NaN fraction in one cutout: 0.0%)
peak  MJy/sr  range : [0.3846, 157.1]
total MJy/sr  range : [2595, 1.051e+04]


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
ax[0].hist(nan_frac[nan_frac > 0], bins=40, color='tab:orange')
ax[0].set(title='NaN fraction (padded cutouts only)', xlabel='fraction of NaN pixels', ylabel='count')
ax[1].hist(np.log10(np.clip(peaks, 1e-30, None)), bins=40)
ax[1].set(title='peak surface brightness', xlabel='log10 peak (MJy/sr)', ylabel='count')
ax[2].hist(np.log10(np.clip(totals, 1e-30, None)), bins=40, color='tab:green')
ax[2].set(title='integrated surface brightness', xlabel='log10 sum (MJy/sr)', ylabel='count')
save_fig(fig, 'cutout_statistics')
plt.show()

## 3. Visualize cutouts

A grid of randomly chosen cutouts (log stretch, NaN padding shown transparent).

In [ ]:
n = 100
idx = rng.choice(len(cutouts), min(n, len(cutouts)), replace=False)
side = int(np.ceil(np.sqrt(len(idx))))
fig, axes = plt.subplots(side, side, figsize=(2 * side, 2 * side), constrained_layout=True)
for ax, i in zip(axes.flat, idx):
    a = np.load(cutouts[i])
    pos = a[np.isfinite(a) & (a > 0)]
    norm = LogNorm(vmin=np.percentile(pos, 5), vmax=np.percentile(pos, 99.9)) if pos.size else None
    ax.imshow(a, norm=norm, origin='lower', interpolation='nearest')
    ax.axis('off')
for ax in axes.flat[len(idx):]:
    ax.axis('off')
plt.suptitle(f'L3 cutouts (random sample of {len(idx)})', fontsize=14)
save_fig(fig, 'cutout_grid_random')
plt.show()

In [ ]:
# group cutouts by system uid so the same systems can be shown across bands
by_uid = defaultdict(dict)
for c in cutouts:
    sca, uid, band = parse_cutout(c)
    by_uid[(sca, uid)][band] = c

# 5x5 grid per band, same systems (those that have every band) in the same order
complete = [k for k, v in by_uid.items() if all(b in v for b in bands)]
print(f'{len(complete):,} systems have all bands {bands}')
chosen = [complete[i] for i in rng.choice(len(complete), min(25, len(complete)), replace=False)]

for band in bands:
    fig, axes = plt.subplots(5, 5, figsize=(12, 12), constrained_layout=True)
    for ax, key in zip(axes.flat, chosen):
        a = np.load(by_uid[key][band])
        pos = a[np.isfinite(a) & (a > 0)]
        norm = LogNorm(vmin=np.percentile(pos, 5), vmax=np.percentile(pos, 99.9)) if pos.size else None
        ax.imshow(a, norm=norm, origin='lower', interpolation='nearest')
        ax.set_title(key[1], fontsize=7)
        ax.axis('off')
    for ax in axes.flat[len(chosen):]:
        ax.axis('off')
    plt.suptitle(f'L3 cutouts - {band}', fontsize=14)
    save_fig(fig, f'cutout_grid_{band}')
    plt.show()

In [ ]:
# RGB composites for systems that have the three target bands
from astropy.visualization import make_lupton_rgb

rgb_bands = ['F158', 'F129', 'F106']   # R, G, B
have_rgb = [k for k, v in by_uid.items() if all(b in v for b in rgb_bands)]
print(f'{len(have_rgb):,} systems have RGB bands {rgb_bands}')

if have_rgb:
    sel = [have_rgb[i] for i in rng.choice(len(have_rgb), min(25, len(have_rgb)), replace=False)]
    fig, axes = plt.subplots(5, 5, figsize=(12, 12), constrained_layout=True)
    for ax, key in zip(axes.flat, sel):
        chans = [np.nan_to_num(np.load(by_uid[key][b]).astype(float)) for b in rgb_bands]
        rgb = make_lupton_rgb(image_r=chans[0], image_g=chans[1], image_b=chans[2],
                              stretch=0.5, Q=7, minimum=float(np.min(chans[0])))
        ax.imshow(rgb, origin='lower', interpolation='nearest')
        ax.set_title(key[1], fontsize=7)
        ax.axis('off')
    for ax in axes.flat[len(sel):]:
        ax.axis('off')
    plt.suptitle(f'L3 Lupton RGB (R={rgb_bands[0]}, G={rgb_bands[1]}, B={rgb_bands[2]})', fontsize=14)
    save_fig(fig, 'cutout_grid_lupton_rgb')
    plt.show()

## 4. Diagnostic mosaic images

The full co-added L3 mosaics the pipeline saved as PNGs — one per (SCA, band, batch). Useful for spotting tiling/co-add artifacts before trusting the cutouts.

In [ ]:
import matplotlib.image as mpimg

show = mosaics[:12]
if not show:
    print('No mosaic PNGs found.')
else:
    cols = 3
    rows = int(np.ceil(len(show) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, png in zip(axes, show):
        ax.imshow(mpimg.imread(png))
        ax.set_title(os.path.basename(png), fontsize=8)
        ax.axis('off')
    for ax in axes[len(show):]:
        ax.axis('off')
    save_fig(fig, 'diagnostic_mosaics')
    plt.show()

## 5. Compare against the single-exposure L2 cutouts (optional)

The L3 mosaic co-adds four dithers, so the same system should look like a deeper version of its `05_romanisim` (L2) counterpart. This pairs them up where both exist. Pixel units differ (L3 is MJy/sr, L2 is DN/s), so each panel is stretched independently.

In [ ]:
l2_dir = os.path.join(config['data_dir'], label, '05_romanisim')
l2_cutouts = glob(os.path.join(l2_dir, 'sca*', 'Exposure_*.npy')) + \
             glob(os.path.join(l2_dir, 'Exposure_*.npy'))

if not l2_cutouts:
    print(f'No L2 cutouts found under {l2_dir}; skipping comparison.')
else:
    l2_by_name = {os.path.basename(p): p for p in l2_cutouts}
    pairs = [(c, l2_by_name[os.path.basename(c)]) for c in cutouts
             if os.path.basename(c) in l2_by_name]
    print(f'{len(pairs):,} systems present in both L3 and L2')

    def show(ax, arr, title):
        pos = arr[np.isfinite(arr) & (arr > 0)]
        norm = LogNorm(vmin=np.percentile(pos, 5), vmax=np.percentile(pos, 99.9)) if pos.size else None
        ax.imshow(arr, norm=norm, origin='lower', interpolation='nearest')
        ax.set_title(title, fontsize=8)
        ax.axis('off')

    sel = [pairs[i] for i in rng.choice(len(pairs), min(6, len(pairs)), replace=False)] if pairs else []
    if sel:
        fig, axes = plt.subplots(len(sel), 2, figsize=(6, 3 * len(sel)), constrained_layout=True)
        axes = np.atleast_2d(axes)
        for row, (l3p, l2p) in zip(axes, sel):
            name = os.path.basename(l3p)[len('Exposure_'):-len('.npy')]
            show(row[0], np.load(l2p),  f'L2  {name}')
            show(row[1], np.load(l3p),  f'L3  {name}')
        save_fig(fig, 'l2_vs_l3_comparison')
        plt.show()